**Cell 1: UPLOAD DATASETS**

This cell handles the initial setup, importing necessary libraries and uploading the global (`geopol_data.xlsx`) and local (`expanded_philippine_data.csv`) datasets into the Colab environment.

In [ ]:
# ==========================================
# CELL 1: UPLOAD DATASETS
# ==========================================
from google.colab import files
import pandas as pd
import numpy as np

print("Please upload: \n1. geopol_data.xlsx\n2. expanded_philippine_data.csv")
uploaded = files.upload()

**Cell 2: DATA PREPARATION**

This cell loads the global Excel data into a pandas DataFrame, defines the 23 numeric features for the model, and cleans the data by converting all columns to a numeric format and removing any rows that are missing a compressive strength value.

In [ ]:
# ==========================================
# CELL 2: DATA PREPARATION
# ==========================================
# 1. Load the Global Training Data
df_train = pd.read_excel('geopol_data.xlsx')

# 2. Define the Feature List (23 numeric predictors)
features = [
    'Precursor_kg_m3', 'Fine_Agg_kg_m3', 'Coarse_Agg_kg_m3',
    'NaOH_kg_m3', 'Na2SiO3_kg_m3', 'Molarity_NaOH',
    'SS_SH_Ratio', 'l_b_Ratio', 'Sand_Binder_Ratio',
    'Si_Al_Ratio', 'SiO2_%', 'Al2O3_%', 'Fe2O3_%', 'CaO_%',
    'MgO_%', 'Na2O_%', 'K2O_%', 'D50_um', 'SSA_m2_kg',
    'Amorphous_Phase_%', 'Curing_Temp_C', 'Curing_Time_h', 'Age_Days'
]

# --- IMPROVEMENT: Centralized dictionary for professional plot labels ---
feature_labels = {
    'Curing_Temp_C': 'Curing Temp (°C)', 'Curing_Time_h': 'Curing Time (h)', 'Age_Days': 'Testing Age (Days)',
    'Molarity_NaOH': 'NaOH Molarity', 'SS_SH_Ratio': 'Silicate/Hydroxide Ratio', 'l_b_Ratio': 'Liquid/Binder Ratio',
    'Sand_Binder_Ratio': 'Sand/Binder Ratio', 'Si_Al_Ratio': 'Si/Al Ratio', 'SiO2_%': 'Silica (SiO2 %)',
    'Al2O3_%': 'Alumina (Al2O3 %)', 'Fe2O3_%': 'Iron Oxide (Fe2O3 %)', 'CaO_%': 'Calcium Oxide (CaO %)',
    'MgO_%': 'Magnesium Oxide (MgO %)', 'Na2O_%': 'Sodium Oxide (Na2O %)', 'K2O_%': 'Potassium Oxide (K2O %)',
    'D50_um': 'Particle Size (D50)', 'SSA_m2_kg': 'Surface Area (m²/kg)',
    'Amorphous_Phase_%': 'Amorphous Phase (%)', 'Precursor_kg_m3': 'Precursor (kg/m³)',
    'Fine_Agg_kg_m3': 'Fine Aggregate (kg/m³)', 'Coarse_Agg_kg_m3': 'Coarse Aggregate (kg/m³)',
    'NaOH_kg_m3': 'NaOH (kg/m³)', 'Na2SiO3_kg_m3': 'Na2SiO3 (kg/m³)'
}

# 3. Clean Data: Force all columns to numeric and handle errors
for col in features + ['Strength_MPa']:
    df_train[col] = pd.to_numeric(df_train[col], errors='coerce')

# 4. Filter out rows that are missing the Target variable
df_train = df_train.dropna(subset=['Strength_MPa'])

X_train = df_train[features]
y_train = df_train['Strength_MPa']

print(f"Data ready! Training on {len(df_train)} rows with {len(features)} features.")

**Cell 3: TRAIN XGBOOST MODEL**

This cell initializes and trains the state-of-the-art XGBoost model on the cleaned global dataset, leveraging its native ability to handle missing values (`NaNs`) for robust learning, and then evaluates its internal performance (R² and MAE).

In [ ]:
# ==========================================
# CELL 3: TRAIN XGBOOST MODEL
# ==========================================
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

# Initialize XGBoost Regressor
# We use 'missing=np.nan' to tell the model to learn from the blanks
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    missing=np.nan
)

# Train the model
xgb_model.fit(X_train, y_train)

# Evaluate on training data
y_pred_train = xgb_model.predict(X_train)
print("="*50)
print("INTERNAL TRAINING PERFORMANCE")
print("="*50)
print(f"R2 Score: {r2_score(y_train, y_pred_train):.4f}")
print(f"MAE: {mean_absolute_error(y_train, y_pred_train):.2f} MPa")

***Cell 3A: EXPLAIN THE ORIGINAL GLOBAL MODEL WITH SHAP ("BEFORE")***

Using the SHAP library, this cell analyzes the trained global model to generate feature importance plots, revealing which chemical and physical parameters the AI learned were most influential for predicting strength based on the global literature data.

In [ ]:
# =================================================================
# CELL 3A: EXPLAIN THE ORIGINAL GLOBAL MODEL WITH SHAP ("BEFORE")
# =================================================================
import shap
import matplotlib.pyplot as plt

print("="*60)
print("Analyzing the 'Thinking' of the Original Global AI Model...")
print("="*60)

# 1. Create a SHAP explainer for the ORIGINAL global model
global_explainer = shap.Explainer(xgb_model)

# 2. Calculate SHAP values on the GLOBAL training data
global_shap_values = global_explainer(X_train)

# 3. Create the SHAP bar plot for the global model
shap.summary_plot(
    global_shap_values,
    X_train,
    plot_type="bar",
    feature_names=[feature_labels.get(f, f) for f in X_train.columns],
    show=False
)
plt.title('Feature Importance (Original Global Model)', fontsize=14, weight='bold')
plt.tight_layout()
plt.savefig('shap_importance_global.png', dpi=300)
plt.show()

# 4. Create the SHAP beeswarm plot for the global model
shap.summary_plot(
    global_shap_values,
    X_train,
    feature_names=[feature_labels.get(f, f) for f in X_train.columns],
    show=False
)
plt.title('SHAP Summary Plot (Original Global Model)', fontsize=14, weight='bold')
plt.tight_layout()
plt.savefig('shap_beeswarm_global.png', dpi=300)
plt.show()

**Cell 4: FINE-TUNING THE AI ON PHILIPPINE DATA (TRANSFER LEARNING**)

This cell implements the transfer learning process by splitting the Philippine samples into a small "fine-tuning" set and a "hold-out" test set, then re-trains the global AI on the tuning data to adapt its knowledge to the local material domain.

In [ ]:
# ===================================================================================
# CELL 4 (IMPROVED): DYNAMIC FINE-TUNING ON PHILIPPINE DATA (TRANSFER LEARNING)
# ===================================================================================
from sklearn.model_selection import train_test_split

# 1. Load the full Philippine dataset
try:
    df_ph = pd.read_csv('expanded_philippine_data.csv')
except FileNotFoundError:
    print("ERROR: 'expanded_philippine_data.csv' not found. Please upload it in Cell 1.")
    # This stops the cell if the file is missing
    raise

# 2. Ensure all columns are numeric and drop rows with missing strength values
for col in features + ['Strength_MPa']:
    df_ph[col] = pd.to_numeric(df_ph[col], errors='coerce')
df_ph = df_ph.dropna(subset=['Strength_MPa'])

print(f"Loaded {len(df_ph)} valid samples from the Philippine dataset.")

# 3. Isolate the full Philippine feature set and target
X_ph_full = df_ph[features]
y_ph_full = df_ph['Strength_MPa']

# --- DYNAMIC SPLITTING LOGIC ---
n_samples = len(df_ph)
if n_samples < 25:
    # For small datasets, use a fixed number of samples for the hold-out test
    test_split_size = 5 if n_samples > 5 else 0.3 # Use 5, unless there are 5 or fewer samples
    split_method = f"fixed number ({test_split_size})"
else:
    # For larger datasets, use a standard percentage
    test_split_size = 0.3
    split_method = f"percentage ({test_split_size*100}%)"

print(f"Dataset size is {n_samples}. Using split method: {split_method}.\n")

# 4. Split the Philippine samples into a "fine-tuning" set and a final "hold-out" test set
X_tune, X_holdout, y_tune, y_holdout = train_test_split(
    X_ph_full,
    y_ph_full,
    test_size=test_split_size,
    random_state=42
)

# 5. Create a NEW XGBoost model for fine-tuning
# The 'xgb_model' parameter initializes the new model with the knowledge of the pre-trained global model
fine_tuned_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    random_state=42,
    missing=np.nan
)
# This is the transfer learning step!
fine_tuned_model.fit(X_tune, y_tune, xgb_model=xgb_model)

print("="*50)
print("AI Model successfully fine-tuned on local Philippine data.")
print(f"Fine-tuning samples: {len(X_tune)}")
print(f"Final validation samples: {len(X_holdout)}")
print("="*50)

***Cell 4A: INSTALL SHAP LIBRARY***

This is a one-time setup cell that installs the `shap` library, which is required for the advanced feature importance analysis performed in the next step.

In [ ]:
# ==========================================
# CELL 4A: INSTALL SHAP LIBRARY
# ==========================================
!pip install shap

***Cell 4B: EXPLAIN THE FINE-TUNED MODEL WITH SHAP ("AFTER")***

This cell performs a SHAP analysis on the fine-tuned model to visualize how the AI's priorities have shifted, showing which features became more or less important after being "re-educated" on the unique chemistry of Philippine materials.

In [ ]:
# ==========================================
# CELL 4B (FIXED): EXPLAIN THE FINE-TUNED MODEL WITH SHAP
# ==========================================
import shap

# --- THE FIX IS HERE: We define the labels dictionary within this cell ---
feature_labels = {
    'Curing_Temp_C': 'Curing Temp (°C)', 'Curing_Time_h': 'Curing Time (h)', 'Age_Days': 'Testing Age (Days)',
    'Molarity_NaOH': 'NaOH Molarity', 'SS_SH_Ratio': 'Silicate/Hydroxide Ratio', 'l_b_Ratio': 'Liquid/Binder Ratio',
    'Sand_Binder_Ratio': 'Sand/Binder Ratio', 'SiO2_%': 'Silica (SiO2 %)', 'Al2O3_%': 'Alumina (Al2O3 %)',
    'Fe2O3_%': 'Iron Oxide (Fe2O3 %)', 'CaO_%': 'Calcium Oxide (CaO %)', 'MgO_%': 'Magnesium Oxide (MgO %)',
    'Na2O_%': 'Sodium Oxide (Na2O %)', 'K2O_%': 'Potassium Oxide (K2O %)', 'D50_um': 'Particle Size (D50)',
    'SSA_m2_kg': 'Surface Area', 'Amorphous_Phase_%': 'Amorphous Phase (%)',
    # Adding the raw mass columns just in case they appear in the importance plot
    'Precursor_kg_m3': 'Precursor (kg/m³)', 'Fine_Agg_kg_m3': 'Fine Aggregate (kg/m³)',
    'Coarse_Agg_kg_m3': 'Coarse Aggregate (kg/m³)', 'NaOH_kg_m3': 'NaOH (kg/m³)', 'Na2SiO3_kg_m3': 'Na2SiO3 (kg/m³)'
}

# 1. Create a SHAP explainer object based on your fine-tuned model
explainer = shap.Explainer(fine_tuned_model)

# 2. Calculate the SHAP values for the data used to fine-tune the model
shap_values = explainer(X_tune)

# 3. Create the SHAP summary plot
print("="*60)
print("What the Fine-Tuned AI Learned About Philippine Materials")
print("="*60)
print("This plot shows the most important features for the fine-tuned model.")
print("Red dots mean a high value for that feature; blue dots mean a low value.")
print("Dots on the right INCREASE the strength prediction; dots on the left DECREASE it.\n")

# Create the simple bar chart
shap.summary_plot(
    shap_values,
    X_tune,
    plot_type="bar",
    feature_names=[feature_labels.get(f, f) for f in X_tune.columns],
    show=False
)
plt.title('Feature Importance (Fine-Tuned Model)', fontsize=14, weight='bold')
plt.tight_layout()
plt.savefig('shap_importance_tuned.png', dpi=300)
plt.show()

# Create the more detailed SHAP beeswarm plot
shap.summary_plot(
    shap_values,
    X_tune,
    feature_names=[feature_labels.get(f, f) for f in X_tune.columns],
    show=False
)
plt.title('SHAP Summary Plot (Fine-Tuned Model)', fontsize=14, weight='bold')
plt.tight_layout()
plt.savefig('shap_beeswarm_tuned.png', dpi=300)
plt.show()

**Cell 5: COMPARING THE GLOBAL vs. FINE-TUNED MODEL**

This cell makes predictions on the hold-out test set using both the original global AI and the newly fine-tuned AI, then calculates and prints a comparison table to quantitatively prove the improvement in accuracy.

In [ ]:
# ==========================================
# CELL 5: COMPARING THE GLOBAL vs. FINE-TUNED MODEL
# ==========================================
# 1. Predict using the original global model
y_pred_global = xgb_model.predict(X_holdout)

# 2. Predict using the NEW fine-tuned model
y_pred_tuned = fine_tuned_model.predict(X_holdout)

# 3. Create a comparison DataFrame
results_df = pd.DataFrame({
    'Actual_Strength': y_holdout,
    'Global_Model_Prediction': y_pred_global,
    'Fine_Tuned_Prediction': y_pred_tuned
})

# Calculate the MAE for both models
mae_global = mean_absolute_error(y_holdout, y_pred_global)
mae_tuned = mean_absolute_error(y_holdout, y_pred_tuned)

print("="*80)
print("FINAL VALIDATION: Comparing AI Performance on Hold-Out Philippine Samples")
print("="*80)
print(results_df.round(2))
print("-" * 80)
print(f"MAE of Original Global Model : {mae_global:.2f} MPa  <-- Demonstrates 'Domain Shift'")
print(f"MAE of Fine-Tuned Model      : {mae_tuned:.2f} MPa  <-- Demonstrates successful 'Transfer Learning'")
print(f"\nCONCLUSION: Fine-tuning improved the model's accuracy by {((mae_global - mae_tuned) / mae_global * 100):.1f}%!")

**Cell 6: PLOT**

This final cell generates the "Actual vs. Predicted" scatter plot, visually demonstrating the project's key finding by plotting the high-error predictions of the original model against the low-error predictions of the fine-tuned model on the same graph.

In [ ]:
# ==========================================
# CELL 6 (FIXED): FINAL PUBLICATION PLOT
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 7))

# Diagonal line for perfect prediction
max_val = max(results_df['Actual_Strength'].max(), results_df['Global_Model_Prediction'].max()) + 5
plt.plot([0, max_val], [0, max_val], 'k--', lw=2, label='Perfect Prediction (1:1)')

# Scatter plot for the ORIGINAL global model predictions
sns.scatterplot(
    x=results_df['Actual_Strength'],
    y=results_df['Global_Model_Prediction'],
    s=100,
    color='black',
    edgecolor='black',
    label='Original Global AI (High Error)',
    marker='o',
    zorder=5,
    alpha=0.8
)

# Scatter plot for the FINE-TUNED model predictions
sns.scatterplot(
    x=results_df['Actual_Strength'],
    y=results_df['Fine_Tuned_Prediction'],
    s=100,
    color='black',
    edgecolor='black',
    label='Fine-Tuned AI (Low Error)',
    marker='s',
    zorder=5,
    alpha=0.8
)

plt.xlim(0, max_val)
plt.ylim(0, max_val)
plt.xlabel('Actual Compressive Strength (MPa)', fontsize=12)
plt.ylabel('AI Predicted Compressive Strength (MPa)', fontsize=12)
plt.title('Effect of Transfer Learning on AI Model Accuracy', fontsize=14, weight='bold')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.savefig('transfer_learning_comparison.png', dpi=300)
plt.show()

In [ ]:
# ==========================================
# CELL: GENERATE TABLE 3 (EVALUATION METRICS)
# ==========================================
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Calculate Metrics for the Original Global Model
mae_global = mean_absolute_error(y_holdout, y_pred_global)
rmse_global = np.sqrt(mean_squared_error(y_holdout, y_pred_global))
r2_global = r2_score(y_holdout, y_pred_global)

# 2. Calculate Metrics for the Fine-Tuned Model
# Note: We apply a physical constraint max(0, prediction) to handle the negative artifact
y_pred_tuned_physical = np.maximum(0, y_pred_tuned)

mae_tuned = mean_absolute_error(y_holdout, y_pred_tuned_physical)
rmse_tuned = np.sqrt(mean_squared_error(y_holdout, y_pred_tuned_physical))
r2_tuned = r2_score(y_holdout, y_pred_tuned_physical)

# 3. Create a Publication-Ready Table (DataFrame)
metrics_data = {
    'Metric': ['Mean Absolute Error (MAE) [MPa]', 'Root Mean Square Error (RMSE) [MPa]', 'R-squared (R²)'],
    'Original Global Model': [f"{mae_global:.2f}", f"{rmse_global:.2f}", f"{r2_global:.3f}"],
    'Fine-Tuned Local Model': [f"{mae_tuned:.2f}", f"{rmse_tuned:.2f}", f"{r2_tuned:.3f}"],
    'Improvement (%)': [
        f"{((mae_global - mae_tuned) / mae_global * 100):.1f}%",
        f"{((rmse_global - rmse_tuned) / rmse_global * 100):.1f}%",
        "N/A" # R2 improvement is non-linear, usually left as N/A in percentage terms
    ]
}

table_3 = pd.DataFrame(metrics_data)

print("="*80)
print("TABLE 3: Predictive Performance Metrics on Philippine Hold-Out Set")
print("="*80)
display(table_3) # 'display' renders a nice HTML table in Colab

In [ ]:
# ==========================================
# CELL: GENERATE FIGURE 4 (RESIDUAL PLOT)
# ==========================================
import matplotlib.pyplot as plt

# 1. Calculate Residuals (Actual - Predicted)
# Positive residual = Model underpredicted; Negative residual = Model overpredicted
residuals_global = y_holdout - y_pred_global
residuals_tuned = y_holdout - y_pred_tuned_physical # Using the physically constrained predictions

# 2. Set up the plot aesthetics
plt.figure(figsize=(10, 7))
plt.style.use('default') # Clean, academic background

# 3. Plot Global Model Residuals
plt.scatter(y_pred_global, residuals_global,
            color='dimgray', marker='o', s=80, alpha=0.7,
            edgecolor='black', label='Original Global AI (Domain Shift)')

# 4. Plot Fine-Tuned Model Residuals
plt.scatter(y_pred_tuned_physical, residuals_tuned,
            color='#007acc', marker='s', s=80, alpha=0.9,
            edgecolor='black', label='Fine-Tuned AI (Adapted)')

# 5. Add the "Perfect Prediction" Zero-Error Line
plt.axhline(y=0, color='black', linestyle='--', linewidth=2, label='Zero Error Line')

# 6. Formatting for Publication
plt.title('Residual Error Analysis: Mitigation of Optimism Bias', fontsize=14, weight='bold')
plt.xlabel('Predicted Compressive Strength (MPa)', fontsize=12)
plt.ylabel('Residual Error (Actual - Predicted) (MPa)', fontsize=12)
plt.legend(fontsize=11, loc='lower right')
plt.grid(True, linestyle='--', alpha=0.5)

# Add text annotation explaining the bias
plt.text(45, -45, "Global model systematically\noverpredicts (negative residuals)",
         fontsize=10, color='dimgray', style='italic',
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

plt.tight_layout()

# 7. Save and Show
plt.savefig('Figure_4_Residual_Plot.png', dpi=300)
plt.show()

print("\nFigure 4 saved as 'Figure_4_Residual_Plot.png' at 300 DPI.")

In [ ]:
# ==========================================
# CELL: GENERATE FIGURE 2 (PCA DOMAIN SHIFT) - FIXED
# ==========================================
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer # <-- NEW IMPORT TO HANDLE NaNs

# 1. Combine the Global and Philippine features for PCA
X_global_pca = df_train[features].copy()
X_local_pca = df_ph[features].copy()

# Add a label column to track which dataset each point belongs to
X_global_pca['Dataset'] = 'Global Literature'
X_local_pca['Dataset'] = 'Philippine Local'

# Concatenate into one DataFrame
df_combined = pd.concat([X_global_pca, X_local_pca], axis=0, ignore_index=True)

# Separate features from labels
X_combined = df_combined[features]
y_dataset = df_combined['Dataset']

# 2. HANDLE MISSING VALUES (NaNs)
# PCA cannot handle blank cells. We fill them with the column's median value.
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_combined)

# 3. Standardize the features (Crucial for PCA so all features have mean=0, variance=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# 4. Apply PCA to reduce 23 dimensions down to 2 dimensions
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Get the variance explained by the two components to put on the axes
explained_variance = pca.explained_variance_ratio_ * 100

# 5. Plotting
plt.figure(figsize=(10, 7))
plt.style.use('default')

# Plot Global Data
mask_global = (y_dataset == 'Global Literature')
plt.scatter(X_pca[mask_global, 0], X_pca[mask_global, 1],
            color='dimgray', alpha=0.4, s=50, label='Global Literature Data', edgecolor='none')

# Plot Philippine Data
mask_local = (y_dataset == 'Philippine Local')
plt.scatter(X_pca[mask_local, 0], X_pca[mask_local, 1],
            color='#e74c3c', alpha=0.9, s=80, marker='^', label='Philippine Data', edgecolor='black')

# 6. Aesthetics and Labels
plt.title('PCA Visualization of Domain Shift', fontsize=14, weight='bold')
plt.xlabel(f'Principal Component 1 ({explained_variance[0]:.1f}% variance)', fontsize=12)
plt.ylabel(f'Principal Component 2 ({explained_variance[1]:.1f}% variance)', fontsize=12)
plt.legend(fontsize=11, loc='best')
plt.grid(True, linestyle='--', alpha=0.5)

# Add a descriptive text box
plt.text(0.02, 0.02,
         "Distinct clustering indicates the local materials\noccupy a different feature space than the global literature.",
         transform=plt.gca().transAxes, fontsize=10, style='italic',
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))

plt.tight_layout()

# 7. Save and Show
plt.savefig('Figure_2_PCA_Domain_Shift.png', dpi=300)
plt.show()

print("\nFigure 2 saved as 'Figure_2_PCA_Domain_Shift.png' at 300 DPI.")

In [ ]:
# ==========================================
# CELL: GENERATE FIGURE 7 (SHAP DEPENDENCE PLOTS)
# ==========================================
import matplotlib.pyplot as plt
import numpy as np
import shap

# 1. Ensure we have the SHAP values from the fine-tuned model
explainer = shap.Explainer(fine_tuned_model)
shap_explanation = explainer(X_tune)
shap_values_raw = shap_explanation.values

# 2. Define the NEW top features we want to investigate
features_to_plot = ['CaO_%', 'Sand_Binder_Ratio', 'Molarity_NaOH']

# Nice display names for the axes
display_names = {
    'CaO_%': 'Calcium Oxide (CaO %)',
    'Sand_Binder_Ratio': 'Sand-to-Binder Ratio',
    'Molarity_NaOH': 'NaOH Molarity (M)'
}

# 3. Set up a 1x3 publication-ready figure
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plt.style.use('default')

# 4. Loop through the features and create a scatter plot for each
for i, feature in enumerate(features_to_plot):
    col_idx = X_tune.columns.get_loc(feature)

    x_vals = X_tune.iloc[:, col_idx]
    y_vals = shap_values_raw[:, col_idx]

    axes[i].scatter(x_vals, y_vals, color='#e74c3c', alpha=0.7, edgecolor='black', s=50)
    axes[i].axhline(0, color='black', linestyle='--', linewidth=1.5, alpha=0.8)

    axes[i].set_xlabel(display_names[feature], fontsize=12, weight='bold')
    axes[i].set_ylabel('SHAP Value (Impact on Strength)', fontsize=12)
    axes[i].grid(True, linestyle='--', alpha=0.4)
    axes[i].set_title(f'Effect of {display_names[feature]}', fontsize=13)

# 5. Add a main title and adjust layout
plt.suptitle('Figure 7: SHAP Dependence Plots for Key Local Features', fontsize=16, weight='bold', y=1.05)
plt.tight_layout()

# 6. Save and Show
plt.savefig('Figure_7_SHAP_Dependence_Updated.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nUpdated Figure 7 saved as 'Figure_7_SHAP_Dependence_Updated.png' at 300 DPI.")

In [ ]:
# ==========================================
# CELL: GENERATE TABLE 2 (DESCRIPTIVE STATS)
# ==========================================
import pandas as pd

# 1. Define the key features we want to summarize for the paper
summary_features = [
    'Strength_MPa', 'Age_Days', 'Curing_Temp_C',
    'Si_Al_Ratio', 'l_b_Ratio', 'Sand_Binder_Ratio',
    'SiO2_%', 'Al2O3_%', 'Fe2O3_%', 'CaO_%'
]

# Nicer names for the table
nice_names = {
    'Strength_MPa': 'Compressive Strength (MPa)', 'Age_Days': 'Testing Age (Days)',
    'Curing_Temp_C': 'Curing Temp (°C)', 'Si_Al_Ratio': 'Si/Al Ratio',
    'l_b_Ratio': 'Liquid/Binder Ratio', 'Sand_Binder_Ratio': 'Sand/Binder Ratio',
    'SiO2_%': 'SiO2 (%)', 'Al2O3_%': 'Al2O3 (%)', 'Fe2O3_%': 'Fe2O3 (%)', 'CaO_%': 'CaO (%)'
}

def get_stats(df, features):
    stats = []
    for feat in features:
        if feat in df.columns:
            s = df[feat].dropna()
            stats.append({
                'Feature': nice_names[feat],
                'Min': s.min(),
                'Max': s.max(),
                'Mean': s.mean(),
                'Std Dev': s.std()
            })
    return pd.DataFrame(stats)

# 2. Calculate stats for both datasets
# (Assuming df_train is your global data and df_ph is your Philippine data)
global_stats = get_stats(df_train, summary_features)
local_stats = get_stats(df_ph, summary_features)

# 3. Merge them together into one publication-ready table
table_2 = pd.merge(global_stats, local_stats, on='Feature', suffixes=(' (Global)', ' (Local)'))

# Format numbers to 2 decimal places
for col in table_2.columns:
    if col != 'Feature':
        table_2[col] = table_2[col].map('{:.2f}'.format)

print("="*100)
print("TABLE 2: Descriptive Statistics of the Global Pre-training and Local Fine-Tuning Datasets")
print("="*100)
display(table_2)

# Optional: Save to CSV so you can easily open it in Excel and copy to Word
table_2.to_csv('Table_2_Descriptive_Statistics.csv', index=False)
print("\nSaved to 'Table_2_Descriptive_Statistics.csv'")

In [ ]:
# ==========================================
# CELL: GENERATE FIGURE 8 (RECALIBRATION SHIFT PLOT)
# ==========================================
import matplotlib.pyplot as plt
import numpy as np

# Ensure predictions are physically valid (no negative strengths)
y_pred_tuned_physical = np.maximum(0, y_pred_tuned)

plt.figure(figsize=(12, 8))
plt.style.use('default')

# 1. Plot the 1:1 Perfect Prediction Line
plt.plot([0, 70], [0, 70], 'k--', linewidth=2, label='Perfect Prediction (1:1 Line)')

# 2. Loop through each hold-out sample to draw the "Recalibration Arrow"
for i in range(len(y_holdout)):
    actual = y_holdout.iloc[i]
    global_pred = y_pred_global[i]
    tuned_pred = y_pred_tuned_physical[i]

    # Draw an arrow from the Global Prediction down to the Fine-Tuned Prediction
    plt.annotate('', xy=(actual, tuned_pred), xytext=(actual, global_pred),
                 arrowprops=dict(arrowstyle="->", color='gray', alpha=0.6, lw=1.5))

# 3. Plot the actual points on top of the arrows
plt.scatter(y_holdout, y_pred_global, color='dimgray', s=60, alpha=0.8,
            edgecolor='black', label='Original Global Prediction (Optimism Bias)')

plt.scatter(y_holdout, y_pred_tuned_physical, color='#007acc', marker='s', s=60, alpha=1.0,
            edgecolor='black', label='Fine-Tuned Prediction (Corrected)')

# 4. Aesthetics and Formatting
plt.title('Visualizing AI Recalibration: Correction of Optimism Bias via Transfer Learning',
          fontsize=14, weight='bold')
plt.xlabel('Actual Compressive Strength (MPa)', fontsize=12)
plt.ylabel('Predicted Compressive Strength (MPa)', fontsize=12)
plt.xlim(-2, 65)
plt.ylim(-2, 80)
plt.legend(fontsize=11, loc='upper left')
plt.grid(True, linestyle='--', alpha=0.4)

# 5. Add explanatory text for the reviewers
plt.text(35, 10,
         "Arrows indicate the exact penalty applied\nby the AI to correct the domain shift.",
         fontsize=11, style='italic',
         bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray'))

plt.tight_layout()

# 6. Save and Show
plt.savefig('Figure_8_Recalibration_Shift.png', dpi=300)
plt.show()

print("\nFigure saved as 'Figure_8_Recalibration_Shift.png' at 300 DPI.")